In [ ]:
# src/ on the path - cwd is notebooks/
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

In [ ]:
# allows change in src/ to be reflected in notebook without restarting kernel
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import config

In [ ]:
from metrics import split_chips, ndvi, metrics, tune_threshold
from prepare import load_chips

chips = load_chips()
train_chips, val_chips, test_chips = split_chips(chips)

In [ ]:
best_t = tune_threshold(train_chips)

nd_test = np.concatenate([ndvi(c["img"]).ravel() for c in test_chips])
truth_test = np.concatenate([c["msk"].ravel() for c in test_chips])
lc8_test = np.concatenate([c["lc8"].ravel() for c in test_chips])

baseline = metrics(nd_test > best_t, truth_test)
baseline

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(nd_test[lc8_test == 1], bins=80, alpha=0.6, density=True, label="tree canopy")
plt.hist(nd_test[lc8_test == 2], bins=80, alpha=0.6, density=True, label="grass/shrub")
plt.axvline(best_t, color="k", ls="--", label=f"threshold {best_t:.2f}")
plt.xlabel("NDVI")
plt.ylabel("density")
plt.legend()

plt.savefig(config.FIGURES / "02_ndvi_histogram.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
names = {1: "tree canopy", 2: "grass/shrub", 3: "bare ground", 4: "water",
         5: "building", 6: "road", 7: "other impervious", 8: "railroad"}

pred_test = nd_test > best_t

for label, mask in [
    ("false positives (called canopy, wasn't)", pred_test & (truth_test == 0)),
    ("false negatives (missed canopy)", ~pred_test & (truth_test == 1)),
]:
    print(label)
    classes, counts = np.unique(lc8_test[mask], return_counts=True)
    for cls, n in zip(classes, counts):
        print(f"  {names[cls]:18s} {100 * n / counts.sum():5.1f}%")
    print()